# Iteration 3 – pretrained transformer + ensemble with iteration 2

> ⚠️ **This iteration was made entirely with AI** (Claude by Anthropic): the idea, the code and the explanations. It is an extra experiment on top of iteration 2.

| | Iteration 3 |
|---|---|
| **Model** | **3a:** pretrained transformer `bge-small` + small neural network. **3b:** 3a combined with the NB-LR of iteration 2 (ensemble) |
| **Libraries** | `sentence-transformers` (Hugging Face), `torch`, `scikit-learn` |
| **How text becomes numbers** | the transformer reads the whole description and returns 384 numbers that describe its meaning |
| **Input** | description + rank + category + cantrip + casting time |
| **Output** | traits, traditions, save, area, damage types, sustained |
| **Data** | same 1,920 Foundry spells and split as iteration 1 |
| **Result** | micro-F1 **0.787** (3b), 3a alone 0.728 |
| **Test: only rank + description** | micro-F1 **0.719** (3b) |
| **Status** | ⚠️ not better than iteration 2 (same score, much more complex) |

## What changed compared to iteration 2

| | Iteration 2 | Iteration 3 |
|---|---|---|
| Text → numbers | bag of words (0/1 per word and word pair) | pretrained transformer `BAAI/bge-small-en-v1.5`: 384 numbers per text |
| Model | NB-LR, one small model per label | small neural network with one shared layer and one head per field (3a) |
| Final answer | NB-LR | average of NB-LR and 3a, the weight per field chosen on the validation set (3b) |

Data cleaning, inputs, outputs, split and scores are the same as in iteration 2.

In [1]:
import copy
import html
import json
import re
from collections import Counter
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import torch
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from torch import nn

ROOT = Path.cwd()
DATA_DIR = ROOT.parents[1] / "data" / "spellsdata"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
SEED = 42
MIN_LABEL_SUPPORT = 5
C = 1.0  # same as iteration 2
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
EMBEDDING_CHUNK_WORDS = 200
FULL_INPUT = ["rank", "category", "is_cantrip", "cast_time"]
ORIGINAL_IDEA = ["rank"]  # the original idea: the user only types the rank and the description
TRADITIONS = ["arcane", "divine", "occult", "primal"]
CLASS_TRAITS = {
    "animist", "bard", "champion", "cleric", "druid", "exemplar", "kineticist", "magus", "monk",
    "necromancer", "oracle", "psychic", "ranger", "sorcerer", "summoner", "thaumaturge", "witch", "wizard",
}

/Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1030 09:40:32.377000 63393 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Stage 1: text cleaning

Foundry descriptions contain inline commands. They are turned into the text a player would see instead of being deleted (iteration 1 deleted them):

| Raw | Cleaned |
|---|---|
| `@UUID[Compendium.pf2e.conditionitems.Item.Prone]` | `prone` |
| `@Damage[2d6[persistent,fire]]` | `2d6 persistent fire` |
| `@Check[reflex\|dc:20\|basic]` | `basic reflex save` |
| `@Template[cone\|distance:30]` | `30-foot cone` |

In [2]:
INLINE_PATTERN = re.compile(r"@(UUID|Damage|Check|Template|Embed|Localize)\[")
HTML_PATTERN = re.compile(r"<[^>]+>")
RANDOM_ID_PATTERN = re.compile(r"(?=.*\d)[A-Za-z0-9]{16}")

def closing_bracket(value: str, start: int) -> int:
    depth = 0
    for index in range(start, len(value)):
        if value[index] == "[":
            depth += 1
        elif value[index] == "]":
            depth -= 1
            if depth == 0:
                return index
    return len(value) - 1

def render_inline(kind: str, body: str, label: str | None) -> str:
    if label:
        return label
    if kind == "UUID":
        name = body.rsplit(".", 1)[-1]
        return " " if RANDOM_ID_PATTERN.fullmatch(name) else name
    if kind == "Damage":
        return " ".join(re.sub(r"[\[\],]", " ", body).split())
    if kind == "Check":
        parts = body.split("|")
        is_basic = any(part.split(":")[0] == "basic" for part in parts[1:])
        return f"basic {parts[0]} save" if is_basic else f"{parts[0]} check"
    if kind == "Template":
        params = dict(part.split(":", 1) for part in body.split("|") if ":" in part)
        shape = params.get("type", body.split("|")[0])
        return f"{params['distance']}-foot {shape}" if "distance" in params else shape
    return " "

def expand_inline(value: str) -> str:
    parts: list[str] = []
    position = 0
    while (match := INLINE_PATTERN.search(value, position)):
        body_end = closing_bracket(value, match.end() - 1)
        body = value[match.end():body_end]
        end, label = body_end, None
        if body_end + 1 < len(value) and value[body_end + 1] == "{":
            label_end = value.find("}", body_end + 1)
            end = len(value) - 1 if label_end == -1 else label_end
            label = value[body_end + 2:end]
        parts.append(value[position:match.start()])
        parts.append(f" {render_inline(match.group(1), body, label)} ")
        position = end + 1
    parts.append(value[position:])
    return "".join(parts)

def clean_text(value: str) -> str:
    value = expand_inline(value or "")
    value = HTML_PATTERN.sub(" ", value)
    return " ".join(html.unescape(value).lower().split())

example_raw = ("Creatures are knocked @UUID[Compendium.pf2e.conditionitems.Item.Prone] and take "
               "@Damage[2d6[persistent,fire]] damage (@Check[reflex|dc:20|basic]) in a @Template[cone|distance:30].")
print("raw:    ", example_raw)
print("cleaned:", clean_text(example_raw))

raw:     Creatures are knocked @UUID[Compendium.pf2e.conditionitems.Item.Prone] and take @Damage[2d6[persistent,fire]] damage (@Check[reflex|dc:20|basic]) in a @Template[cone|distance:30].
cleaned: creatures are knocked prone and take 2d6 persistent fire damage ( basic reflex save ) in a 30-foot cone .


## Stage 2: dataset and inputs

The same 1,920 spells as iteration 1 (every JSON file with a description and at least one trait or tradition). Besides the description, each spell has fields that a Game Master chooses in Foundry anyway:

* **rank** (1-10);
* **category**: spell, focus, ritual or impossible (the folder of the file);
* **is_cantrip**: cantrips have rank 1 in Foundry, so the rank alone does not show it;
* **cast_time**: 1/2/3 actions, reaction, free, minutes/hours (exploration) or days (downtime).

The model is trained twice: once with **all of these inputs** (`FULL_INPUT`) and once with **only rank + description** (`ORIGINAL_IDEA`, the test at the end).

In [3]:
def spell_category(path: Path) -> str:
    top_folder = path.relative_to(DATA_DIR).parts[0]
    return {"spells": "spell", "focus": "focus", "rituals": "ritual", "impossible-spells": "impossible"}[top_folder]

def cast_time_bucket(value: Any) -> str:
    text = str(value or "").strip().lower()
    if text in {"1", "2", "3", "reaction", "free"}:
        return text
    if re.fullmatch(r"\d (to|or) \d.*", text):
        return "variable"
    if "minute" in text or "hour" in text:
        return "exploration"
    if "day" in text or "week" in text:
        return "downtime"
    return "other"

def area_bucket(value: str | None) -> str:
    if not value:
        return "none"
    return value if value in {"burst", "emanation", "cone", "line"} else "other"

def load_spells(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(data_dir.rglob("*.json")):
        if path.name == "_folders.json":
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        system = payload.get("system", {})
        traits_data = system.get("traits", {})
        raw_text = system.get("description", {}).get("value", "")
        traits = sorted({str(label).casefold() for label in traits_data.get("value", []) if label})
        traditions = sorted({str(label).casefold() for label in traits_data.get("traditions", []) if label})
        if not raw_text.strip() or not (traits or traditions):
            continue
        defense = system.get("defense") or {}
        rows.append({
            "name": payload.get("name", path.stem),
            "raw_text": raw_text,
            "text": clean_text(raw_text),
            "rank": int(system.get("level", {}).get("value", 0) or 0),
            "category": spell_category(path),
            "is_cantrip": "cantrip" in traits,
            "cast_time": cast_time_bucket(system.get("time", {}).get("value")),
            "traits": traits,
            "traditions": traditions,
            "save": (defense.get("save") or {}).get("statistic") or "none",
            "area": area_bucket((system.get("area") or {}).get("type")),
            "damage_types": sorted({entry.get("type") for entry in (system.get("damage") or {}).values() if entry.get("type")}),
            "sustained": bool((system.get("duration") or {}).get("sustained")),
        })
    return pd.DataFrame(rows)

df = load_spells(DATA_DIR)
print(f"spells={len(df)}")
print(f"categories: {dict(Counter(df['category']))}")
print(f"cantrips: {int(df['is_cantrip'].sum())}")

spells=1920
categories: {'focus': 545, 'impossible': 5, 'ritual': 93, 'spell': 1277}
cantrips: 120


## Stage 3: what the model predicts

| Field | Foundry field | Type |
|---|---|---|
| `traits` | `system.traits.value` | several labels |
| `traditions` | `system.traits.traditions` | several labels (arcane, divine, occult, primal) |
| `save` | `system.defense.save.statistic` | one choice: none / fortitude / reflex / will |
| `area` | `system.area.type` | one choice: none / burst / emanation / cone / line / other |
| `damage_types` | `system.damage[*].type` | several labels |
| `sustained` | `system.duration.sustained` | yes / no |

Not predicted: class traits (`witch`, `cleric`, ...), because the description does not say which class gets a focus spell, and traits with fewer than 5 spells. Every field is stored as a 0/1 matrix (one column per label), so all fields are handled the same way.

In [4]:
trait_support = Counter(trait for labels in df["traits"] for trait in labels)
trait_labels = sorted(trait for trait, count in trait_support.items() if count >= MIN_LABEL_SUPPORT and trait not in CLASS_TRAITS)
damage_support = Counter(label for labels in df["damage_types"] for label in labels)
damage_labels = sorted(label for label, count in damage_support.items() if count >= MIN_LABEL_SUPPORT)

GROUPS = {
    "traits": trait_labels,
    "traditions": TRADITIONS,
    "save": ["none", "fortitude", "reflex", "will"],
    "area": ["none", "burst", "emanation", "cone", "line", "other"],
    "damage_types": damage_labels,
    "sustained": ["sustained"],
}
ONE_CHOICE = {"save", "area"}  # exactly one answer per spell: the most likely label is taken

def targets(frame: pd.DataFrame, group: str) -> np.ndarray:
    # 0/1 matrix: one row per spell, one column per label of the group.
    def labels_of(value: Any) -> list[str]:
        if group == "sustained":
            return ["sustained"] if value else []
        return [value] if group in ONE_CHOICE else value
    return np.array([[label in labels_of(value) for label in GROUPS[group]] for value in frame[group]], dtype=np.int64)

print({group: len(labels) for group, labels in GROUPS.items()})
print(f"class traits (chosen by the user): {sorted(set(trait_support) & CLASS_TRAITS)}")

{'traits': 68, 'traditions': 4, 'save': 4, 'area': 6, 'damage_types': 16, 'sustained': 1}
class traits (chosen by the user): ['animist', 'bard', 'champion', 'cleric', 'druid', 'magus', 'monk', 'necromancer', 'oracle', 'psychic', 'ranger', 'sorcerer', 'summoner', 'witch', 'wizard']


## Stage 4: split and scores

* **Same split as iteration 1** (`sample(frac=1, random_state=42)`, 80/10/10): 1,536 training, 192 validation and the same **192 test spells**.
* For fields with several labels, the decision threshold is chosen on the validation set. The test set is only used for the final numbers.
* **Main score** (the one in the README): micro-F1 of traits + traditions in the 100 labels of iteration 1. Class traits and rare traits count as misses, so it can be compared with iterations 0 and 1.
* Reference: always answering `concentrate, manipulate` scores 0.473.

In [5]:
def split_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_start = int(len(shuffled) * 0.8)
    valid_start = int(len(shuffled) * 0.9)
    return shuffled.iloc[:test_start], shuffled.iloc[test_start:valid_start], shuffled.iloc[valid_start:]

train_df, valid_df, test_df = split_frame(df)
frames = {"train": train_df, "valid": valid_df, "test": test_df}
Y = {split: {group: targets(frame, group) for group in GROUPS} for split, frame in frames.items()}
ITERATION1_LABELS = sorted({label for row in df.itertuples() for label in [*row.traits, *row.traditions]})

def f1_scores(predicted: np.ndarray, actual: np.ndarray) -> dict[str, float]:
    predicted, actual = np.asarray(predicted).astype(bool), np.asarray(actual).astype(bool)
    tp = (predicted & actual).sum(axis=0)
    fp = (predicted & ~actual).sum(axis=0)
    fn = (~predicted & actual).sum(axis=0)
    per_label = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    return {"micro_f1": float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
            "macro_f1": float(per_label[actual.sum(axis=0) > 0].mean()),
            "precision": float(tp.sum() / max(tp.sum() + fp.sum(), 1)),
            "recall": float(tp.sum() / max(tp.sum() + fn.sum(), 1))}

THRESHOLD_GRID = np.round(np.concatenate([[0.02, 0.03], np.arange(0.05, 0.96, 0.05)]), 2)

def tune_threshold(probabilities: np.ndarray, actual: np.ndarray) -> float:
    return float(max(THRESHOLD_GRID, key=lambda threshold: f1_scores(probabilities >= threshold, actual)["micro_f1"]))

def decide(probabilities: dict[str, np.ndarray], thresholds: dict[str, float]) -> dict[str, np.ndarray]:
    # 0/1 per label: the most likely label for save and area, every label above the threshold otherwise.
    return {group: np.eye(p.shape[1], dtype=int)[p.argmax(axis=1)] if group in ONE_CHOICE else (p >= thresholds[group]).astype(int)
            for group, p in probabilities.items()}

def labels_per_spell(decided: dict[str, np.ndarray]) -> list[dict[str, list[str]]]:
    return [{group: [label for label, on in zip(GROUPS[group], decided[group][row]) if on] for group in GROUPS}
            for row in range(len(decided["traits"]))]

def label_matrix_100(label_sets: list[set[str]]) -> np.ndarray:
    return np.array([[label in labels for label in ITERATION1_LABELS] for labels in label_sets], dtype=np.int64)

def evaluate(probabilities: dict[str, np.ndarray], thresholds: dict[str, float], split: str = "test") -> dict[str, float]:
    decided = decide(probabilities, thresholds)
    predicted = [{*spell["traits"], *spell["traditions"]} for spell in labels_per_spell(decided)]
    actual = [{*row.traits, *row.traditions} for row in frames[split].itertuples()]
    report = {"overall": f1_scores(label_matrix_100(predicted), label_matrix_100(actual))["micro_f1"]}
    for group in GROUPS:
        if group in ONE_CHOICE:
            report[group] = float((decided[group] == Y[split][group]).all(axis=1).mean())
        else:
            report[group] = f1_scores(decided[group], Y[split][group])["micro_f1"]
    return {key: round(value, 4) for key, value in report.items()}

SCORE_ROWS = {"overall": "traits + traditions, 100 labels (micro-F1)", "traits": "traits (micro-F1)",
              "traditions": "traditions (micro-F1)", "save": "save (accuracy)", "area": "area (accuracy)",
              "damage_types": "damage types (micro-F1)", "sustained": "sustained (F1)"}

def score_table(reports: dict[str, dict[str, float]]) -> pd.DataFrame:
    return pd.DataFrame({name: [report[key] for key in SCORE_ROWS] for name, report in reports.items()},
                        index=list(SCORE_ROWS.values())).round(3)

constant = label_matrix_100([{"concentrate", "manipulate"}] * len(test_df))
actual_100 = label_matrix_100([{*row.traits, *row.traditions} for row in test_df.itertuples()])
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")
print(f"reference, always [concentrate, manipulate]: {f1_scores(constant, actual_100)['micro_f1']:.3f}")

train=1536, valid=192, test=192
reference, always [concentrate, manipulate]: 0.473


## Stage 5: the iteration 2 model (NB-LR)

The ensemble needs the probabilities of iteration 2. So that this notebook runs on its own, NB-LR is trained again here with **exactly the same code and settings** as in `iteration_2.ipynb` (a few seconds).

In [6]:
vectorizer = CountVectorizer(token_pattern=r"[a-z0-9][a-z0-9'+-]*", ngram_range=(1, 2), min_df=2, binary=True)
vectorizer.fit(train_df["text"])

def nb_log_ratio(X: sparse.csr_matrix, y: np.ndarray, alpha: float = 1.0) -> np.ndarray:
    # Naive Bayes: how much more often each feature occurs in spells WITH the label than WITHOUT it (log ratio).
    p = alpha + np.asarray(X[y == 1].sum(axis=0)).ravel()
    q = alpha + np.asarray(X[y == 0].sum(axis=0)).ravel()
    return np.log((p / p.sum()) / (q / q.sum()))

def features(frame: pd.DataFrame, encoder: OneHotEncoder) -> sparse.csr_matrix:
    # 0/1 for every word and word pair + the extra inputs (rank, category, ...) as 0/1 columns.
    extra = encoder.transform(frame[encoder.feature_names_in_].astype(str))
    return sparse.hstack([vectorizer.transform(frame["text"]), extra]).tocsr().astype(np.float64)

def fit_nb_lr(X: sparse.csr_matrix, y: np.ndarray) -> tuple[np.ndarray, LogisticRegression]:
    ratio = nb_log_ratio(X, y)
    return ratio, LogisticRegression(C=C, max_iter=3000).fit(X.multiply(ratio).tocsr(), y)

def predict_nb_lr(models: list[tuple[np.ndarray, LogisticRegression]], X: sparse.csr_matrix) -> np.ndarray:
    return np.column_stack([model.predict_proba(X.multiply(ratio).tocsr())[:, 1] for ratio, model in models])

def train_nb_lr(inputs: list[str]) -> tuple[dict[str, Any], dict[str, dict[str, np.ndarray]]]:
    encoder = OneHotEncoder(handle_unknown="ignore").fit(train_df[inputs].astype(str))
    X = {split: features(frame, encoder) for split, frame in frames.items()}
    models = {group: [fit_nb_lr(X["train"], y) for y in Y["train"][group].T] for group in GROUPS}
    probabilities = {split: {group: predict_nb_lr(models[group], X[split]) for group in GROUPS} for split in ["valid", "test"]}
    thresholds = {group: tune_threshold(probabilities["valid"][group], Y["valid"][group]) for group in GROUPS if group not in ONE_CHOICE}
    return {"encoder": encoder, "models": models, "thresholds": thresholds}, probabilities

## Stage 6: pretrained transformer (3a)

`BAAI/bge-small-en-v1.5` is a small transformer (33M parameters) that was already trained by others on a lot of English text. It is **not trained further** here. It turns a text into 384 numbers that describe its meaning, so similar descriptions get similar numbers even when they use different words.

Long descriptions are split into pieces of 200 words, and the vectors of the pieces are averaged. The first run downloads the model (~130 MB).

In [7]:
def embed_texts(texts: list[str]) -> np.ndarray:
    pieces, owners = [], []
    for index, text in enumerate(texts):
        words = text.split()
        for start in range(0, max(len(words), 1), EMBEDDING_CHUNK_WORDS):
            pieces.append(" ".join(words[start:start + EMBEDDING_CHUNK_WORDS]))
            owners.append(index)
    piece_vectors = sentence_encoder.encode(pieces, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    vectors = np.zeros((len(texts), piece_vectors.shape[1]), dtype=np.float32)
    np.add.at(vectors, np.array(owners), piece_vectors)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

sentence_encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = {split: embed_texts(frame["text"].tolist()) for split, frame in frames.items()}
print(f"embeddings: {embeddings['train'].shape} (spells x numbers)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9991.46it/s]

embeddings: (1536, 384) (spells x numbers)


On top of the 384 numbers (+ the extra inputs as 0/1 columns) a small network predicts all 6 fields:

* one shared hidden layer (256), then one output head per field;
* fields with several labels use a sigmoid per label, `save` and `area` a softmax;
* rare labels get a higher weight in the loss (√(negatives / positives), at most 10);
* training stops when the validation loss has not improved for 15 epochs (early stopping).

In [8]:
class MultiTaskNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, dropout: float = 0.3) -> None:
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.heads = nn.ModuleDict({group: nn.Linear(hidden_dim, len(labels)) for group, labels in GROUPS.items()})

    def forward(self, features: torch.Tensor) -> dict[str, torch.Tensor]:
        shared = self.shared(features)
        return {group: head(shared) for group, head in self.heads.items()}

TARGETS = {split: {group: torch.tensor(Y[split][group].argmax(axis=1)) if group in ONE_CHOICE
                   else torch.tensor(Y[split][group], dtype=torch.float32) for group in GROUPS} for split in frames}
POS_WEIGHTS = {group: torch.tensor(np.sqrt((len(train_df) - counts) / np.maximum(counts, 1)).clip(1, 10), dtype=torch.float32)
               for group, counts in ((group, Y["train"][group].sum(axis=0)) for group in GROUPS if group not in ONE_CHOICE)}

def network_loss(outputs: dict[str, torch.Tensor], targets: dict[str, torch.Tensor]) -> torch.Tensor:
    loss = torch.zeros(())
    for group, logits in outputs.items():
        if group in ONE_CHOICE:
            loss = loss + nn.functional.cross_entropy(logits, targets[group])
        else:
            loss = loss + nn.functional.binary_cross_entropy_with_logits(logits, targets[group], pos_weight=POS_WEIGHTS[group])
    return loss

def network_probabilities(outputs: dict[str, torch.Tensor]) -> dict[str, np.ndarray]:
    return {group: (logits.softmax(dim=1) if group in ONE_CHOICE else logits.sigmoid()).numpy() for group, logits in outputs.items()}

def network_input(frame: pd.DataFrame, vectors: np.ndarray, encoder: OneHotEncoder) -> torch.Tensor:
    extra = encoder.transform(frame[encoder.feature_names_in_].astype(str)).toarray()
    return torch.tensor(np.hstack([vectors, extra]), dtype=torch.float32)

def train_network(inputs: list[str], max_epochs: int = 300, patience: int = 15, batch_size: int = 64):
    torch.manual_seed(SEED)
    encoder = OneHotEncoder(handle_unknown="ignore").fit(train_df[inputs].astype(str))
    X = {split: network_input(frame, embeddings[split], encoder) for split, frame in frames.items()}
    network = MultiTaskNetwork(X["train"].shape[1])
    optimizer = torch.optim.AdamW(network.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)
    best_loss, best_state, best_epoch = float("inf"), None, 0
    for epoch in range(1, max_epochs + 1):
        network.train()
        order = torch.randperm(len(train_df))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            loss = network_loss(network(X["train"][batch]), {group: value[batch] for group, value in TARGETS["train"].items()})
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        network.eval()
        with torch.no_grad():
            valid_loss = network_loss(network(X["valid"]), TARGETS["valid"]).item()
        scheduler.step(valid_loss)
        if valid_loss < best_loss - 1e-4:
            best_loss, best_state, best_epoch = valid_loss, copy.deepcopy(network.state_dict()), epoch
        elif epoch - best_epoch >= patience:
            break
    network.load_state_dict(best_state)
    network.eval()
    with torch.no_grad():
        probabilities = {split: network_probabilities(network(X[split])) for split in ["valid", "test"]}
    print(f"  network: stopped after {epoch} epochs, best epoch {best_epoch} (validation loss {best_loss:.3f})")
    return {"encoder": encoder, "network": network}, probabilities

## Stage 7: ensemble (3b) and results

3b averages the probabilities of NB-LR (iteration 2) and the network (3a). For every field, the weight of NB-LR (0, 0.25, 0.5, 0.75 or 1) is chosen on the validation set.

In [9]:
WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]

def validation_score(group: str, probabilities: np.ndarray) -> float:
    if group in ONE_CHOICE:
        return float((probabilities.argmax(axis=1) == Y["valid"][group].argmax(axis=1)).mean())
    return f1_scores(probabilities >= tune_threshold(probabilities, Y["valid"][group]), Y["valid"][group])["micro_f1"]

def mix(nb_lr: dict[str, np.ndarray], network: dict[str, np.ndarray], weights: dict[str, float]) -> dict[str, np.ndarray]:
    return {group: weights[group] * nb_lr[group] + (1 - weights[group]) * network[group] for group in GROUPS}

def run_iteration3(inputs: list[str]) -> dict[str, Any]:
    nb_lr, nb_lr_probabilities = train_nb_lr(inputs)
    network, network_probs = train_network(inputs)
    network_thresholds = {group: tune_threshold(network_probs["valid"][group], Y["valid"][group]) for group in GROUPS if group not in ONE_CHOICE}
    weights = {group: max(WEIGHTS, key=lambda weight: validation_score(
        group, weight * nb_lr_probabilities["valid"][group] + (1 - weight) * network_probs["valid"][group])) for group in GROUPS}
    mixed = {split: mix(nb_lr_probabilities[split], network_probs[split], weights) for split in ["valid", "test"]}
    thresholds = {group: tune_threshold(mixed["valid"][group], Y["valid"][group]) for group in GROUPS if group not in ONE_CHOICE}
    reports = {"2 – NB-LR": evaluate(nb_lr_probabilities["test"], nb_lr["thresholds"]),
               "3a – transformer + network": evaluate(network_probs["test"], network_thresholds),
               "3b – ensemble of 2 and 3a": evaluate(mixed["test"], thresholds)}
    return {"nb_lr": nb_lr, "network": network, "weights": weights, "thresholds": thresholds, "reports": reports}

full = run_iteration3(FULL_INPUT)
print(f"  weight of NB-LR per field (1 = only NB-LR, 0 = only the network): {full['weights']}")
score_table(full["reports"])

  network: stopped after 68 epochs, best epoch 53 (validation loss 2.441)
  weight of NB-LR per field (1 = only NB-LR, 0 = only the network): {'traits': 1.0, 'traditions': 0.5, 'save': 0.5, 'area': 0.25, 'damage_types': 1.0, 'sustained': 1.0}


,2 – NB-LR,3a – transformer + network,3b – ensemble of 2 and 3a
"traits + traditions, 100 labels (micro-F1)",0.786,0.728,0.787
traits (micro-F1),0.819,0.729,0.819
traditions (micro-F1),0.781,0.788,0.784
save (accuracy),0.896,0.797,0.854
area (accuracy),0.833,0.807,0.812
damage types (micro-F1),0.713,0.314,0.713
sustained (F1),0.595,0.384,0.595


## Test: the original idea – only rank + description

The original idea of the project: **the user types only the rank and the description of a spell, and the model fills in everything else.**

The whole iteration (NB-LR, network and ensemble) is trained again with only rank + description as input: no category, no cantrip, no casting time.

In [10]:
original = run_iteration3(ORIGINAL_IDEA)
print(f"  weight of NB-LR per field: {original['weights']}")
score_table({"3b, full input": full["reports"]["3b – ensemble of 2 and 3a"],
             "3b, only rank + description": original["reports"]["3b – ensemble of 2 and 3a"],
             "3a alone, only rank + description": original["reports"]["3a – transformer + network"]})

  network: stopped after 71 epochs, best epoch 56 (validation loss 2.781)
  weight of NB-LR per field: {'traits': 1.0, 'traditions': 0.75, 'save': 0.5, 'area': 0.25, 'damage_types': 1.0, 'sustained': 1.0}


,"3b, full input","3b, only rank + description","3a alone, only rank + description"
"traits + traditions, 100 labels (micro-F1)",0.787,0.719,0.641
traits (micro-F1),0.819,0.777,0.677
traditions (micro-F1),0.784,0.667,0.623
save (accuracy),0.854,0.875,0.797
area (accuracy),0.812,0.823,0.807
damage types (micro-F1),0.713,0.724,0.352
sustained (F1),0.595,0.556,0.269


### What the model makes of rank + description

`predict_spell(rank, description)` is the original idea as a function: the final model 3b, trained on rank + description only. Two homebrew spells and three spells from the test set (with the correct Foundry answer below them).

In [11]:
HOMEBREW = [
    ("Lava Geyser (homebrew, project proposal)", 4,
     "You channel magical energy to rupture the earth, summoning a burst of molten rock. The eruption deals massive heat damage "
     "to anyone in the area and requires you to perform complex hand gestures to sustain it. Each creature in the area must "
     "attempt a basic Reflex save."),
    ("Molten Burst (homebrew, also used in iteration 1)", 3,
     "A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage."),
]
REAL_SPELLS = ["Gust of Wind", "Suggestion", "Wordsmith"]  # from the test set; Wordsmith is a focus spell

def as_text(spell: dict[str, list[str]]) -> dict[str, str]:
    return {group: ", ".join(labels) or "–" for group, labels in spell.items()}

def example_table(predict_spell) -> pd.DataFrame:
    rows = [{"spell": name, "rank": rank, **as_text(predict_spell(rank, description))} for name, rank, description in HOMEBREW]
    for name in REAL_SPELLS:
        spell = test_df[test_df["name"] == name]
        rank = int(spell["rank"].iloc[0])
        rows.append({"spell": f"{name} (test set)", "rank": rank, **as_text(predict_spell(rank, spell["raw_text"].iloc[0]))})
        rows.append({"spell": "↳ correct answer", "rank": rank, **as_text(labels_per_spell({group: targets(spell, group) for group in GROUPS})[0])})
    return pd.DataFrame(rows).set_index("spell")

In [12]:
def predict_spell(rank: int, description: str) -> dict[str, list[str]]:
    frame = pd.DataFrame([{"text": clean_text(description), "rank": rank}])
    nb_lr, network = original["nb_lr"], original["network"]
    X = features(frame, nb_lr["encoder"])
    nb_lr_probabilities = {group: predict_nb_lr(nb_lr["models"][group], X) for group in GROUPS}
    with torch.no_grad():
        outputs = network["network"](network_input(frame, embed_texts(frame["text"].tolist()), network["encoder"]))
    probabilities = mix(nb_lr_probabilities, network_probabilities(outputs), original["weights"])
    return labels_per_spell(decide(probabilities, original["thresholds"]))[0]

example_table(predict_spell)

,rank,traits,traditions,save,area,damage_types,sustained
spell,,,,,,,
"Lava Geyser (homebrew, project proposal)",4,"concentrate, earth, focus, manipulate","arcane, divine, primal",reflex,burst,–,–
"Molten Burst (homebrew, also used in iteration 1)",3,"concentrate, fire, focus, manipulate","arcane, occult, primal",reflex,burst,fire,–
Gust of Wind (test set),1,"air, concentrate, manipulate","arcane, primal",fortitude,burst,bludgeoning,–
↳ correct answer,1,"air, concentrate, manipulate","arcane, primal",fortitude,line,bludgeoning,–
Suggestion (test set),4,"concentrate, incapacitation, linguistic, manip...","arcane, occult",will,none,–,–
↳ correct answer,4,"concentrate, incapacitation, linguistic, manip...","arcane, occult",will,none,–,–
Wordsmith (test set),4,"concentrate, focus, manipulate","arcane, occult",none,none,–,–
↳ correct answer,4,"concentrate, focus, manipulate",–,none,none,–,–


## Saved model

In [13]:
torch.save({"full_input": full["network"]["network"].state_dict(),
            "only_rank_and_description": original["network"]["network"].state_dict()}, ARTIFACT_DIR / "network.pt")
# Everything else the app needs: one-hot encoders of the network, ensemble weights and thresholds.
joblib.dump({variant: {"network_encoder": run["network"]["encoder"], "weights": run["weights"], "thresholds": run["thresholds"]}
             for variant, run in {"full_input": full, "only_rank_and_description": original}.items()}, ARTIFACT_DIR / "ensemble.joblib")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps({
    "full_input": full["reports"], "only_rank_and_description": original["reports"],
    "ensemble_weights": {"full_input": full["weights"], "only_rank_and_description": original["weights"]}}, indent=2), encoding="utf-8")
print(f"saved network.pt, ensemble.joblib and metrics.json (the NB-LR part is saved in iteration_2/artifacts)")

saved network.pt, ensemble.joblib and metrics.json (the NB-LR part is saved in iteration_2/artifacts)


## Conclusions

* **The transformer alone (3a) is weaker than NB-LR:** 0.728 vs 0.786. Spell traits and damage types depend on exact game words (*disbelieve*, *persistent fire*, *battle form*), and the bag of words of NB-LR catches them directly. The transformer is only a little better for traditions.
* **The ensemble (3b) scores 0.787, the same as iteration 2 (0.786).** It is slightly better for traditions, but worse for save and area. The weights chosen on 192 validation spells do not carry over to the test spells.
* **Test: only rank + description → 0.719** (iteration 2: 0.720). Same picture as in iteration 2: what is written in the description works, but focus, cantrip and the traditions of focus spells cannot be read from the text.
* **Decision:** iteration 3 needs PyTorch, a 130 MB model and much more code for no gain, so **iteration 2 stays the final model**. A transformer would only pay off if it is fine-tuned on the spells, which needs more data or more compute.